In [1]:
# Cell 1: Imports and helper functions

import os
import json
import pandas as pd
import numpy as np

# Percentile helper: ignores NaNs and returns None if series is empty
def pct(series, q):
    s = pd.to_numeric(series, errors="coerce").dropna()
    if s.empty:
        return None
    return float(np.percentile(s, q, method="linear"))

print("✓ Imports loaded successfully")

✓ Imports loaded successfully


In [2]:
# Cell 2: CONFIG — PASTE FULL PATHS ONLY

# ============================================================
# EDIT THESE THREE LINES - Paste complete paths
# ============================================================

# Database name (for display in output CSV)
db_name = "AstraDB"

# Full path to the INPUT CSV file (reads_A1.csv)
a1_csv_path = r"C:\Users\avyaa\logs\reads_A1.csv"

# Full path to the OUTPUT FOLDER (where you want a1_metrics.csv saved)
# You must create this folder manually before running
a1_out_dir = r"C:\Users\avyaa\Astra_results\A1\Data"

# ============================================================
# DO NOT EDIT BELOW
# ============================================================

# Output file paths
a1_metrics_out = os.path.join(a1_out_dir, "a1_metrics.csv")
a1_validation_out = os.path.join(a1_out_dir, "a1_validation.csv")

# A1 expectations
EXPECTED_COLD = 1
EXPECTED_WARM = 10

# Verification
print("Database:", db_name)
print("Input CSV:", a1_csv_path)
print("Output metrics:", a1_metrics_out)
print("Output validation:", a1_validation_out)

if not os.path.exists(a1_csv_path):
    print("\n⚠️ INPUT FILE NOT FOUND! Check your path.")
else:
    print("\n✓ Input file found!")
    
if not os.path.exists(a1_out_dir):
    print("⚠️ OUTPUT FOLDER NOT FOUND! Create it manually first.")
else:
    print("✓ Output folder exists!")

Database: AstraDB
Input CSV: C:\Users\avyaa\logs\reads_A1.csv
Output metrics: C:\Users\avyaa\Astra_results\A1\Data\a1_metrics.csv
Output validation: C:\Users\avyaa\Astra_results\A1\Data\a1_validation.csv

✓ Input file found!
✓ Output folder exists!


In [3]:
# Cell 3: Load A1 CSV, retain A1 rows, and prep columns

# Required columns per logging strategy
required_cols = [
    "timestamp_utc","run_id","db_system","server_version","driver",
    "connection_params","db_name","mode","param_source","operation_type",
    "query_id","complexity","query_name","run_number","is_cold",
    "execution_ms","row_count","error_code","error_message","params_json"
]

df = pd.read_csv(a1_csv_path)

# Check for missing columns
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Keep only A1 reads
df = df[(df["mode"] == "A1") & (df["operation_type"] == "read")].copy()

# Normalize booleans and numerics
df["is_cold"] = df["is_cold"].astype(str).str.lower().isin(["1","true","yes"])
df["run_number"] = pd.to_numeric(df["run_number"], errors="coerce")
df["execution_ms"] = pd.to_numeric(df["execution_ms"], errors="coerce")
df["row_count"] = pd.to_numeric(df["row_count"], errors="coerce")

# Separate views for metrics (exclude error rows with no execution_ms)
df_ok = df[df["execution_ms"].notna()].copy()
df_err = df[df["execution_ms"].isna()].copy()

print("A1 total rows:", len(df))
print("Usable rows (with execution_ms):", len(df_ok))
print("Error rows (no execution_ms):", len(df_err))

A1 total rows: 110
Usable rows (with execution_ms): 110
Error rows (no execution_ms): 0


In [4]:
# Cell 4: Count validation (1 cold + EXPECTED_WARM warm per query)

val_rows = []
for qid, g in df.groupby("query_id"):
    cold_cnt = int((g["is_cold"] == True).sum())
    warm_cnt = int((g["is_cold"] == False).sum())
    errs     = int(g["execution_ms"].isna().sum())
    val_rows.append({
        "db_system": db_name,
        "query_id": qid,
        "cold_count": cold_cnt,
        "warm_count": warm_cnt,
        "error_rows": errs,
        "expected_cold": EXPECTED_COLD,
        "expected_warm": EXPECTED_WARM,
        "valid_counts": (cold_cnt == EXPECTED_COLD) and (warm_cnt >= EXPECTED_WARM)
    })

val_df = pd.DataFrame(val_rows).sort_values("query_id")
val_df.to_csv(a1_validation_out, index=False)

print("✓ Validation saved to:", a1_validation_out)
print("\nValidation summary:")
display(val_df)

✓ Validation saved to: C:\Users\avyaa\Astra_results\A1\Data\a1_validation.csv

Validation summary:


,db_system,query_id,cold_count,warm_count,error_rows,expected_cold,expected_warm,valid_counts
0,AstraDB,R1,1,10,0,1,10,True
1,AstraDB,R10,1,10,0,1,10,True
2,AstraDB,R2,1,10,0,1,10,True
3,AstraDB,R3,1,10,0,1,10,True
4,AstraDB,R4,1,10,0,1,10,True
5,AstraDB,R5,1,10,0,1,10,True
6,AstraDB,R6,1,10,0,1,10,True
7,AstraDB,R7,1,10,0,1,10,True
8,AstraDB,R8,1,10,0,1,10,True
9,AstraDB,R9,1,10,0,1,10,True


In [5]:
# Cell 5: Warm-run percentiles per query (p50, p95, p99, max)

warm = df_ok[df_ok["is_cold"] == False].copy()
agg_warm = []

for qid, g in warm.groupby(["query_id","complexity","query_name"]):
    p50 = pct(g["execution_ms"], 50)
    p95 = pct(g["execution_ms"], 95)
    p99 = pct(g["execution_ms"], 99)
    mx  = float(np.max(g["execution_ms"])) if len(g) else None
    agg_warm.append({
        "query_id": qid[0],
        "complexity": qid[1],
        "query_name": qid[2],
        "warm_p50_ms": p50,
        "warm_p95_ms": p95,
        "warm_p99_ms": p99,
        "warm_max_ms": mx,
        "warm_run_count": int(len(g))
    })

warm_df = pd.DataFrame(agg_warm).sort_values("query_id")

print("✓ Warm percentiles computed")
display(warm_df.head(10))

✓ Warm percentiles computed


,query_id,complexity,query_name,warm_p50_ms,warm_p95_ms,warm_p99_ms,warm_max_ms,warm_run_count
0,R1,Basic,Product by ID,202.0980,224.37035,226.21247,226.673,10
1,R10,Advanced,Attribute-like filter (client-side sort),531.4330,2018.18970,2349.88434,2432.808,10
2,R2,Basic,Product by SKU,200.2615,211.94075,213.53375,213.932,10
3,R3,Basic,Customer order history,197.1130,221.56495,225.80899,226.870,10
4,R4,Basic,Category browse,203.1940,223.25175,230.36715,232.146,10
5,R5,Moderate,Search with filters (equality + range),192.2155,235.93655,242.29451,243.884,10
6,R6,Moderate,Best Sellers 30 Day,400528.8220,413278.28690,413818.82618,413953.961,10
7,R7,Moderate,New arrivals by window,198.3595,207.19130,207.74066,207.878,10
8,R8,Moderate,Order detail with items,402.6650,427.51650,434.79210,436.611,10
9,R9,Advanced,Brand facet count (client-side),207.8005,240.34825,241.05925,241.237,10


In [6]:
# Cell 6: Cold run per query (single sample expected)

cold = df_ok[df_ok["is_cold"] == True].copy()

# If multiple cold rows exist, take the first by run_number ascending
cold = cold.sort_values(["query_id","run_number"], ascending=[True, True])
cold_pick = cold.groupby(["query_id","complexity","query_name"], as_index=False).first()

cold_df = cold_pick.loc[:, ["query_id","complexity","query_name","execution_ms"]].rename(
    columns={"execution_ms":"cold_ms"}
).sort_values("query_id")

print("✓ Cold measurements extracted")
display(cold_df.head(10))

✓ Cold measurements extracted


,query_id,complexity,query_name,cold_ms
0,R1,Basic,Product by ID,261.756
1,R10,Advanced,Attribute-like filter (client-side sort),412.042
2,R2,Basic,Product by SKU,199.927
3,R3,Basic,Customer order history,193.335
4,R4,Basic,Category browse,217.571
5,R5,Moderate,Search with filters (equality + range),217.272
6,R6,Moderate,Best Sellers 30 Day,412752.501
7,R7,Moderate,New arrivals by window,222.902
8,R8,Moderate,Order detail with items,403.669
9,R9,Advanced,Brand facet count (client-side),253.657


In [7]:
# Cell 7: Merge warm + cold and derive extras

metrics = pd.merge(warm_df, cold_df, on=["query_id","complexity","query_name"], how="left")

# Delta and improvement
metrics["delta_p50_ms"] = metrics["warm_p50_ms"] - metrics["cold_ms"]
metrics["pct_impr_p50"] = np.where(
    metrics["cold_ms"].notna() & (metrics["cold_ms"] > 0),
    (metrics["cold_ms"] - metrics["warm_p50_ms"]) / metrics["cold_ms"] * 100.0,
    np.nan
)

# Tail amplification on warm runs: p99 / p50
metrics["tail_amp_p99_p50"] = np.where(
    metrics["warm_p50_ms"].notna() & (metrics["warm_p50_ms"] > 0),
    metrics["warm_p99_ms"] / metrics["warm_p50_ms"],
    np.nan
)

# Attach db_system for clarity and join validation counts
metrics.insert(0, "db_system", db_name)

metrics = pd.merge(
    metrics,
    val_df.loc[:, ["query_id","cold_count","warm_count","valid_counts"]],
    on="query_id",
    how="left"
)

# Order columns
col_order = [
    "db_system","query_id","complexity","query_name",
    "warm_run_count","cold_count","warm_count","valid_counts",
    "warm_p50_ms","warm_p95_ms","warm_p99_ms","warm_max_ms",
    "cold_ms","delta_p50_ms","pct_impr_p50","tail_amp_p99_p50"
]
metrics = metrics.loc[:, col_order]

# Save
metrics.to_csv(a1_metrics_out, index=False)

print("✓ Metrics saved to:", a1_metrics_out)
print("\nFinal metrics:")
display(metrics)

✓ Metrics saved to: C:\Users\avyaa\Astra_results\A1\Data\a1_metrics.csv

Final metrics:


,db_system,query_id,complexity,query_name,warm_run_count,cold_count,warm_count,valid_counts,warm_p50_ms,warm_p95_ms,warm_p99_ms,warm_max_ms,cold_ms,delta_p50_ms,pct_impr_p50,tail_amp_p99_p50
0,AstraDB,R1,Basic,Product by ID,10,1,10,True,202.0980,224.37035,226.21247,226.673,261.756,-59.6580,22.791455,1.119321
1,AstraDB,R10,Advanced,Attribute-like filter (client-side sort),10,1,10,True,531.4330,2018.18970,2349.88434,2432.808,412.042,119.3910,-28.975444,4.421789
2,AstraDB,R2,Basic,Product by SKU,10,1,10,True,200.2615,211.94075,213.53375,213.932,199.927,0.3345,-0.167311,1.066275
3,AstraDB,R3,Basic,Customer order history,10,1,10,True,197.1130,221.56495,225.80899,226.870,193.335,3.7780,-1.954121,1.145581
4,AstraDB,R4,Basic,Category browse,10,1,10,True,203.1940,223.25175,230.36715,232.146,217.571,-14.3770,6.607958,1.133730
5,AstraDB,R5,Moderate,Search with filters (equality + range),10,1,10,True,192.2155,235.93655,242.29451,243.884,217.272,-25.0565,11.532319,1.260536
6,AstraDB,R6,Moderate,Best Sellers 30 Day,10,1,10,True,400528.8220,413278.28690,413818.82618,413953.961,412752.501,-12223.6790,2.961503,1.033181
7,AstraDB,R7,Moderate,New arrivals by window,10,1,10,True,198.3595,207.19130,207.74066,207.878,222.902,-24.5425,11.010444,1.047294
8,AstraDB,R8,Moderate,Order detail with items,10,1,10,True,402.6650,427.51650,434.79210,436.611,403.669,-1.0040,0.248719,1.079786
9,AstraDB,R9,Advanced,Brand facet count (client-side),10,1,10,True,207.8005,240.34825,241.05925,241.237,253.657,-45.8565,18.078153,1.160051


In [8]:
# Cell 8: Sanity checks and summaries

print("=" * 60)
print("A1 METRICS COMPUTATION COMPLETE")
print("=" * 60)

print("\n📁 Files saved:")
print("  -", a1_metrics_out)
print("  -", a1_validation_out)

print("\n⚠️ Queries missing expected counts:")
invalid = val_df[~val_df["valid_counts"]]
if len(invalid) > 0:
    display(invalid)
else:
    print("  ✓ All queries have valid counts!")

print("\n🔥 Largest tail amplification (top 5):")
display(metrics.sort_values("tail_amp_p99_p50", ascending=False).head(5))

print("\n⚡ Biggest cold→warm improvements (top 5 by %):")
display(metrics.sort_values("pct_impr_p50", ascending=False).head(5))

A1 METRICS COMPUTATION COMPLETE

📁 Files saved:
  - C:\Users\avyaa\Astra_results\A1\Data\a1_metrics.csv
  - C:\Users\avyaa\Astra_results\A1\Data\a1_validation.csv

⚠️ Queries missing expected counts:
  ✓ All queries have valid counts!

🔥 Largest tail amplification (top 5):


,db_system,query_id,complexity,query_name,warm_run_count,cold_count,warm_count,valid_counts,warm_p50_ms,warm_p95_ms,warm_p99_ms,warm_max_ms,cold_ms,delta_p50_ms,pct_impr_p50,tail_amp_p99_p50
1,AstraDB,R10,Advanced,Attribute-like filter (client-side sort),10,1,10,True,531.4330,2018.18970,2349.88434,2432.808,412.042,119.3910,-28.975444,4.421789
5,AstraDB,R5,Moderate,Search with filters (equality + range),10,1,10,True,192.2155,235.93655,242.29451,243.884,217.272,-25.0565,11.532319,1.260536
9,AstraDB,R9,Advanced,Brand facet count (client-side),10,1,10,True,207.8005,240.34825,241.05925,241.237,253.657,-45.8565,18.078153,1.160051
3,AstraDB,R3,Basic,Customer order history,10,1,10,True,197.1130,221.56495,225.80899,226.870,193.335,3.7780,-1.954121,1.145581
4,AstraDB,R4,Basic,Category browse,10,1,10,True,203.1940,223.25175,230.36715,232.146,217.571,-14.3770,6.607958,1.133730



⚡ Biggest cold→warm improvements (top 5 by %):


,db_system,query_id,complexity,query_name,warm_run_count,cold_count,warm_count,valid_counts,warm_p50_ms,warm_p95_ms,warm_p99_ms,warm_max_ms,cold_ms,delta_p50_ms,pct_impr_p50,tail_amp_p99_p50
0,AstraDB,R1,Basic,Product by ID,10,1,10,True,202.0980,224.37035,226.21247,226.673,261.756,-59.6580,22.791455,1.119321
9,AstraDB,R9,Advanced,Brand facet count (client-side),10,1,10,True,207.8005,240.34825,241.05925,241.237,253.657,-45.8565,18.078153,1.160051
5,AstraDB,R5,Moderate,Search with filters (equality + range),10,1,10,True,192.2155,235.93655,242.29451,243.884,217.272,-25.0565,11.532319,1.260536
7,AstraDB,R7,Moderate,New arrivals by window,10,1,10,True,198.3595,207.19130,207.74066,207.878,222.902,-24.5425,11.010444,1.047294
4,AstraDB,R4,Basic,Category browse,10,1,10,True,203.1940,223.25175,230.36715,232.146,217.571,-14.3770,6.607958,1.133730
